# Customer Churn Baseline Notebook

Educational baseline for the customer-churn dataset. The dataset has only 12 rows, so results are illustrative and not suitable for real-world deployment.


## 1. Dataset and target

Target: `churned` (`1` = churned, `0` = did not churn). `customer_id` is an identifier and is not used as a feature.


In [ ]:
import pandas as pd
from io import StringIO

csv_data = '''customer_id,tenure_months,support_tickets,monthly_spend,login_count,plan_type,churned
C001,2,4,499,21,Basic,1
C002,18,1,1299,2,Pro,0
C003,6,3,799,14,Basic,1
C004,30,0,1499,1,Pro,0
C005,11,2,999,17,Standard,0
C006,1,5,499,30,Basic,1
C007,24,1,999,3,Standard,1
C008,8,4,799,17,Basic,1
C009,15,2,1299,4,Pro,1
C010,4,3,499,12,Basic,1
C011,20,0,999,4,Standard,0
C012,7,2,799,9,Standard,0
'''
df = pd.read_csv(StringIO(csv_data))
df


## 2. Basic quality checks


In [ ]:
print('Shape:', df.shape)
print('\nMissing values:')
print(df.isna().sum())
print('\nDuplicate customer IDs:', df['customer_id'].duplicated().sum())
print('\nTarget distribution:')
print(df['churned'].value_counts())


## 3. Non-ML baseline

Rule: predict churn when `support_tickets >= 3` OR `tenure_months <= 7`. This is a transparent benchmark only.


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

df['baseline_prediction'] = ((df['support_tickets'] >= 3) | (df['tenure_months'] <= 7)).astype(int)
y_true = df['churned']
y_pred = df['baseline_prediction']
print('Accuracy :', round(accuracy_score(y_true,y_pred),3))
print('Precision:', round(precision_score(y_true,y_pred,zero_division=0),3))
print('Recall   :', round(recall_score(y_true,y_pred,zero_division=0),3))
print('F1       :', round(f1_score(y_true,y_pred,zero_division=0),3))
print('\nConfusion matrix:\n', confusion_matrix(y_true,y_pred))


## 4. Optional ML comparison

A logistic-regression model is included as an educational comparison. With only 12 observations, training-set results are not reliable evidence of deployment performance.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

features=['tenure_months','support_tickets','monthly_spend','login_count','plan_type']
X=df[features]; y=df['churned']
pre=ColumnTransformer([('num',StandardScaler(),['tenure_months','support_tickets','monthly_spend','login_count']),('cat',OneHotEncoder(handle_unknown='ignore'),['plan_type'])])
model=Pipeline([('preprocess',pre),('classifier',LogisticRegression(max_iter=1000))])
model.fit(X,y)
ml_pred=model.predict(X)
print('Illustrative training accuracy:', round(accuracy_score(y,ml_pred),3))
print('Illustrative training precision:', round(precision_score(y,ml_pred,zero_division=0),3))
print('Illustrative training recall:', round(recall_score(y,ml_pred,zero_division=0),3))
print('Illustrative training F1:', round(f1_score(y,ml_pred,zero_division=0),3))


## 5. Responsible interpretation

- The dataset is very small, so validation results are uncertain.
- The non-ML baseline provides a transparent benchmark.
- False positives may cause unnecessary retention actions.
- False negatives may cause missed retention opportunities.
- Low-confidence or high-impact cases should receive human review.
- Real deployment would require larger representative data, leakage checks, appropriate evaluation, monitoring, and rollback procedures.

## 6. Conclusion

This notebook demonstrates baseline modeling for the responsible ML exercise. It is educational evidence, not a production-ready churn model.
